In [20]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [22]:
class FocusStateCNN(nn.Module):
    def __init__(self):
        super(FocusStateCNN, self).__init__()

        # feature extractor convolution
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        #sumamrizer
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        #Judge
        self.fc1 = nn.Linear(16 * 32 * 32, 2)
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = x.view(-1, 16 * 32 * 32)
        x = self.fc1(x)
        return x

In [23]:
import cv2
import os

def extract_frames(video_path, output_folder, frame_skip=3):
    os.makedirs(output_folder, exist_ok=True)

    # 1. Using the exact absolute path for Colab
    colab_path = f"/content/{video_path}"
    cap = cv2.VideoCapture(colab_path)

    # 2. Safety check: did the file actually open?
    if not cap.isOpened():
        print(f"⚠️ Error: OpenCV could not open {colab_path}.")
        return

    count = 0
    saved_count = 0

    while cap.isOpened():
        # 3. THE MISSING LINE! This grabs the actual image array.
        success, frame = cap.read()

        # If we run out of frames, stop the loop
        if not success:
            break

        # Save every 3rd frame
        if count % frame_skip == 0:
            filename = os.path.join(output_folder, f"frame_{saved_count:04d}.jpg")
            cv2.imwrite(filename, frame)
            saved_count += 1

        count += 1

    cap.release()
    print(f"✅ Extracted {saved_count} frames from {video_path} into {output_folder}")

# Run the extraction
extract_frames('focused2.mov', 'dataset/focused', frame_skip=3)
extract_frames('distracted2.mov', 'dataset/distracted', frame_skip=3)

✅ Extracted 103 frames from focused2.mov into dataset/focused
✅ Extracted 106 frames from distracted2.mov into dataset/distracted


In [24]:
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader

# 1. Define the Transforms (Resize to 64x64 and convert to Tensor)
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

# 2. Load the Dataset using ImageFolder
# It will look at 'dataset/' and automatically find the 'focused' and 'distracted' folders.
dataset = datasets.ImageFolder(root='dataset', transform=transform)

# 3. Create the DataLoader
# We will feed the model 16 images at a time (batch_size=16)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

# Let's do a quick sanity check to make sure it worked!
print(f"Classes found: {dataset.classes}")
print(f"Total images loaded: {len(dataset)}")

# Grab one batch of images to check the shape
images, labels = next(iter(dataloader))
print(f"Batch Image Shape: {images.shape}") # Should print [16, 3, 64, 64]
print(f"Batch Label Shape: {labels.shape}") # Should print [16]

Classes found: ['distracted', 'focused']
Total images loaded: 263
Batch Image Shape: torch.Size([16, 3, 64, 64])
Batch Label Shape: torch.Size([16])


In [25]:
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#Create Model
model = FocusStateCNN().to(device)


#Define the grader and the adjuster
criterion = nn.CrossEntropyLoss() #BCEL
optimizer = optim.Adam(model.parameters(), lr=0.001)

#number of itmes to read the whole textbook the whole dataset
num_epochs = 10

for epoch in range(num_epochs):
	running_loss = 0.0
	correct_guesses = 0
	total_images = 0

	for images, labels in dataloader:
		images, labels = images.to(device), labels.to(device)

		optimizer.zero_grad()

		outputs = model(images)

		loss = criterion(outputs, labels)

		loss.backward()

		optimizer.step()

		running_loss += loss.item()

		_, predicted = torch.max(outputs.data, 1)
		total_images += labels.size(0)
		correct_guesses  += (predicted == labels).sum().item()

	epoch_loss = running_loss / len(dataloader)
	epoch_acc = 100 * correct_guesses / total_images

In [26]:
#Save the weights of the newly trained model

torch.save(model.state_dict(), 'focus_model.pth')
print("model weights saved as 'focus_model.pth'")

#Now the model weights needs to be tested to see if it works properly
#Take a picture and make whats called an [inference]



model weights saved as 'focus_model.pth'


In [31]:
import torch
import torchvision.transforms as transforms
from PIL import Image

#MAKE SURE TO ADD THIS FILE PATH
image_path = 'test2.jpg'

#Using RGB as that i shwta the image is tested on
image = Image.open(image_path).convert('RGB')

#same transform code is used from before
transform = transforms.Compose([
    transforms.Resize((64 , 64)),
    transforms.ToTensor()
])

#pytorch expects a bathc so we turn the [3, 64, 64] to [1, 3 64, 64]
input_tensor = transform(image).unsqueeze(0).to(device)

#initialize a blueprint to take the weights we just got and load them
model = FocusStateCNN().to(device)
model.load_state_dict(torch.load('focus_model.pth'))

#this turns off the
model.eval()

with torch.no_grad():
    output = model(input_tensor)

    _, predicted_indesx = torch.max(output, 1)

# the predicted classes 0 is distraected and 1 is focused
classes = ['distracted', 'focused']
predicted_class = classes[predicted_indesx.item()]

#Print prediction made
print(f"The image is predicted to be {predicted_class}")




The image is predicted to be distracted
